# Pointing and Alignment Errors

In this notebook we show how to produce Pointing errors drawn from a normal distribution.

In [1]:
import numpy as np
import pandas as pd
from astropy.io import fits

In [23]:
#path = '/lhome/nicholas/Nextcloud/Platoman/PIC/PIC1.1.0/'
#filename = 'hlsp_aspic_gaia_astrometric-photometric_all-sky_multi_v1.1_cat.fits'
#hdul = fits.open(path + filename)
#hdul[1].header

In [24]:
#data = hdul[1].data
#len(data)

In [7]:
# Please select your input catalogs
inputFileTar = '../../platonium/pic/PIC1.1.0/PIC110target.csv'
inputFileCon = '../../platonium/pic/PIC1.1.0/PIC110contaminant.csv'

# Check that the columns are the correct ones seen below or change them!
cols_tar = [0,    # PIC ID
            3,    # RA
            5,    # Dec 
            53]   # Vmag

cols_con = [2,    # PIC ID
            6,    # RA
            8,    # Dec 
            19]   # Vmag

In [8]:
# Load targets
data = np.genfromtxt(inputFileTar, delimiter=',', usecols=cols_tar)                                                                                                                                      
ID     = data[:,0]  
ra     = data[:,1]                                                                                                                                                                                  
dec    = data[:,2]                                                                                                                                                                                  
mag    = data[:,3]
field  = np.loadtxt(inputFileTar, delimiter=',', usecols=[68], dtype=str) 

# Create pandas data array                                                                                                                                                                                  
d = {'ID': ID, 'ra': ra, 'dec': dec, 'mag': mag, 'field': field}                                                                                                                                          
df = pd.DataFrame(d, columns=['ID', 'ra', 'dec', 'mag', 'field'])

In [9]:
# Load targets
pic_con = np.loadtxt(inputFileCon, delimiter=',', usecols=cols_con)
ID_con   = pic_con[:,0]                                                                                                                                                                   
ra_con   = pic_con[:,1]                                                                                                                                                                   
dec_con  = pic_con[:,2]                                                                                                                                                                   
mag_con  = pic_con[:,3]

# Create pandas data array                                                                                                                                                                                  
d = {'ID': ID_con, 'ra': ra_con, 'dec': dec_con, 'mag': mag_con}                                                                                                                                          
dc = pd.DataFrame(d, columns=['ID', 'ra', 'dec', 'mag'])

In [10]:
# Select field
df = df[df['field'] == 'S']
# We don't need this column anymore
del df['field']

In [11]:
# Select contaminants within the FOV
ra_min,  ra_max  = min(df['ra'])-2,  max(df['ra'])+2
dec_min, dec_max = min(df['dec'])-2, max(df['dec'])+2
dc = dc[(dc['ra']  > ra_min)  & (dc['ra']  < ra_max)]
dc = dc[(dc['dec'] > dec_min) & (dc['dec'] < dec_max)]

In [13]:
# Combine targets and contaminants
df.append(dc)

# Remove contaminants that are also targets - hence doublets
df.drop_duplicates(subset=['ID'])

# Order after RA
df.sort_values(by=['ra'])

,ID,ra,dec,mag
24025,5895252.0,44.807123,-59.122671,11.694930
27255,6431663.0,44.842693,-58.089255,12.270261
22976,5712791.0,44.849542,-59.465446,10.783483
22734,5670033.0,44.901390,-59.546264,11.455007
28414,6630359.0,44.919039,-57.705092,12.953218
...,...,...,...,...
32602,7276460.0,129.950994,-56.409179,10.740518
33830,7459412.0,129.957898,-56.037272,11.049641
37097,7940509.0,129.988349,-55.059097,11.863532
35489,7698833.0,129.994287,-55.546029,11.773412


In [14]:
# Save data as txt and npy
catalog = np.transpose([df['ra'].to_numpy(), df['dec'].to_numpy(), df['mag'].to_numpy()])
np.savetxt('starcatalog.txt', catalog, fmt=['%0.6f', '%0.6f', '%0.3f'])
np.save('starcatalog.txt', catalog)